In [13]:
import pandas as pd

pd.options.future.infer_string = False

df = pd.read_csv('../data/raw/properties.csv')
print("Shape:", df.shape)
df.head()

Shape: (5000, 12)


,property_id,location_id,property_type,year_built,square_footage,bedrooms,bathrooms,lot_size_sqft,garage_spaces,has_pool,has_basement,stories
0,PRP-0001,LOC-0152,Land,1974,807,2,5.5,475670.0,4,True,False,2
1,PRP-0002,LOC-1139,SFR,1995,3879,3,2.0,31430.0,2,False,False,3
2,PRP-0003,LOC-2139,SINGLE-FAMILY,1961,603,5,5.0,12452.0,0,False,True,2
3,PRP-0004,LOC-3428,Town House,1920,4020,1,2.0,14983.0,3,True,True,2
4,PRP-0005,LOC-1391,Single Family Home,1962,504,2,2.0,25789.0,2,False,True,1


In [14]:
# Data type verification
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   property_id     5000 non-null   object 
 1   location_id     5000 non-null   object 
 2   property_type   5000 non-null   object 
 3   year_built      5000 non-null   int64  
 4   square_footage  5000 non-null   int64  
 5   bedrooms        5000 non-null   int64  
 6   bathrooms       5000 non-null   float64
 7   lot_size_sqft   3908 non-null   float64
 8   garage_spaces   5000 non-null   int64  
 9   has_pool        5000 non-null   bool   
 10  has_basement    5000 non-null   bool   
 11  stories         5000 non-null   int64  
dtypes: bool(2), float64(2), int64(5), object(3)
memory usage: 400.5+ KB


In [15]:
# Null value checks
print("Null counts per column:")
print(df.isnull().sum())
print(f"\nTotal nulls: {df.isnull().sum().sum()}")
print(f"Null % per column:")
print((df.isnull().sum() / len(df) * 100).round(2).astype(str) + '%')

Null counts per column:
property_id          0
location_id          0
property_type        0
year_built           0
square_footage       0
bedrooms             0
bathrooms            0
lot_size_sqft     1092
garage_spaces        0
has_pool             0
has_basement         0
stories              0
dtype: int64

Total nulls: 1092
Null % per column:
property_id         0.0%
location_id         0.0%
property_type       0.0%
year_built          0.0%
square_footage      0.0%
bedrooms            0.0%
bathrooms           0.0%
lot_size_sqft     21.84%
garage_spaces       0.0%
has_pool            0.0%
has_basement        0.0%
stories             0.0%
dtype: object


In [16]:
# Duplicate detection
full_dupes = df.duplicated().sum()
print(f"Full-row duplicates: {full_dupes}")
if full_dupes > 0:
    print(df[df.duplicated(keep=False)].sort_values(df.columns.tolist()))

Full-row duplicates: 0


In [17]:
# Validate property_id: format (PRP-XXXX) and uniqueness
invalid_format = df[~df['property_id'].str.match(r'^PRP-\d{4}$')]
duplicate_ids  = df[df['property_id'].duplicated(keep=False)]

print(f"Invalid format (not PRP-XXXX): {len(invalid_format)}")
if not invalid_format.empty:
    print(invalid_format[['property_id']].to_string())

print(f"\nDuplicate property_id: {len(duplicate_ids)}")
if not duplicate_ids.empty:
    print(duplicate_ids[['property_id']].sort_values('property_id').to_string())

Invalid format (not PRP-XXXX): 0

Duplicate property_id: 0


In [18]:
# Validate location_id: format (LOC-XXXX) — foreign key to locations table
invalid_loc_format = df[~df['location_id'].str.match(r'^LOC-\d{4}$')]

print(f"Invalid location_id format (not LOC-XXXX): {len(invalid_loc_format)}")
if not invalid_loc_format.empty:
    print(invalid_loc_format[['property_id', 'location_id']].to_string())

Invalid location_id format (not LOC-XXXX): 0


In [19]:
# Inspect all unique property_type values and their counts
print(df['property_type'].value_counts().to_string())

property_type
Single Family Home    403
Single-Family         389
SINGLE-FAMILY         385
single family         381
sfr                   377
SFR                   351
Condo                 284
Condominium           269
CONDO                 258
condo                 243
condominium           242
TH                    157
TOWNHOUSE             145
Town House            140
Townhouse             138
townhouse             132
MULTI-FAMILY          100
Multi-Family           96
multi family           95
Multi Family           94
Multifamily            77
Lot                    54
LAND                   53
land                   50
lot                    49
Land                   38


In [20]:
# Standardize property_type into 5 canonical categories
type_map = {
    # Single Family
    'Single Family Home': 'Single Family', 'Single-Family': 'Single Family',
    'SINGLE-FAMILY': 'Single Family', 'single family': 'Single Family',
    'sfr': 'Single Family', 'SFR': 'Single Family',
    # Condo
    'Condo': 'Condo', 'Condominium': 'Condo', 'CONDO': 'Condo',
    'condo': 'Condo', 'condominium': 'Condo',
    # Townhouse
    'TH': 'Townhouse', 'TOWNHOUSE': 'Townhouse', 'Town House': 'Townhouse',
    'Townhouse': 'Townhouse', 'townhouse': 'Townhouse',
    # Multi-Family
    'MULTI-FAMILY': 'Multi-Family', 'Multi-Family': 'Multi-Family',
    'multi family': 'Multi-Family', 'Multi Family': 'Multi-Family',
    'Multifamily': 'Multi-Family',
    # Land
    'Lot': 'Land', 'LAND': 'Land', 'land': 'Land', 'lot': 'Land', 'Land': 'Land',
}

df['property_type'] = df['property_type'].map(type_map)

# Verify — should show exactly 5 categories with no nulls
print("Standardized property_type counts:")
print(df['property_type'].value_counts())
print(f"\nUnmapped (NaN): {df['property_type'].isnull().sum()}")

Standardized property_type counts:
property_type
Single Family    2286
Condo            1296
Townhouse         712
Multi-Family      462
Land              244
Name: count, dtype: int64

Unmapped (NaN): 0


In [21]:
# Convert property_type to category dtype
df['property_type'] = df['property_type'].astype('category')

print(f"dtype : {df['property_type'].dtype}")
print(f"Categories : {list(df['property_type'].cat.categories)}")

dtype : category
Categories : ['Condo', 'Land', 'Multi-Family', 'Single Family', 'Townhouse']


In [22]:
# Handle lot_size_sqft: null out outliers first, then fill all nulls with group median

# Step 1: Set outliers (< 100 sqft) to NaN so they don't skew the median
outliers = (df['lot_size_sqft'] < 100).sum()
df.loc[df['lot_size_sqft'] < 100, 'lot_size_sqft'] = None
print(f"Outliers set to NaN (< 100 sqft) : {outliers}")
print(f"Total nulls before fill : {df['lot_size_sqft'].isnull().sum()}")

# Step 2: Fill all nulls with median grouped by property_type + bedrooms
df['lot_size_sqft'] = df.groupby(['property_type', 'bedrooms'])['lot_size_sqft'].transform(
    lambda x: x.fillna(x.median())
)

print(f"Null count after fill : {df['lot_size_sqft'].isnull().sum()}")

Outliers set to NaN (< 100 sqft) : 11
Total nulls before fill : 1103
Null count after fill : 0


C:\Users\dell\AppData\Local\Temp\ipykernel_18448\3854855304.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['lot_size_sqft'] = df.groupby(['property_type', 'bedrooms'])['lot_size_sqft'].transform(


In [23]:
# Validate year_built range (1800–2025)
print(f"year_built stats:\n{df['year_built'].describe()}\n")

invalid_year = df[(df['year_built'] < 1800) | (df['year_built'] > 2025)]
print(f"Invalid year_built (outside 1800–2025): {len(invalid_year)}")
if not invalid_year.empty:
    print(invalid_year[['property_id', 'property_type', 'year_built']].to_string())

year_built stats:
count    5000.000000
mean     1970.989400
std        29.847021
min      1920.000000
25%      1945.000000
50%      1971.000000
75%      1997.000000
max      2023.000000
Name: year_built, dtype: float64

Invalid year_built (outside 1800–2025): 0


In [24]:
# Validate numeric columns for realistic ranges
numeric_rules = {
    'square_footage':  (100,    20_000),
    'bedrooms':        (0,      20),
    'bathrooms':       (0,      20),
    'garage_spaces':   (0,      10),
    'stories':         (1,      10),
    'lot_size_sqft':   (100,    10_000_000),
}

print(f"{'Column':<20} {'Min':>10} {'Max':>12} {'Invalid':>10}")
print("-" * 56)
for col, (lo, hi) in numeric_rules.items():
    invalid = df[(df[col] < lo) | (df[col] > hi)]
    print(f"{col:<20} {lo:>10,} {hi:>12,} {len(invalid):>10}")

print("\n--- Flagged rows per column ---")
for col, (lo, hi) in numeric_rules.items():
    invalid = df[(df[col] < lo) | (df[col] > hi)]
    if not invalid.empty:
        print(f"\n{col} (outside {lo}–{hi:,}):")
        print(invalid[['property_id', 'property_type', col]].to_string())

Column                      Min          Max    Invalid
--------------------------------------------------------
square_footage              100       20,000          0
bedrooms                      0           20          0
bathrooms                     0           20          0
garage_spaces                 0           10          0
stories                       1           10          0
lot_size_sqft               100   10,000,000          0

--- Flagged rows per column ---


In [25]:
# Export cleaned data
df.to_csv('../cleaned/properties_cleaned.csv', index=False)

print(f"Exported: cleaned/properties_cleaned.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Exported: cleaned/properties_cleaned.csv
Shape: (5000, 12)
Columns: ['property_id', 'location_id', 'property_type', 'year_built', 'square_footage', 'bedrooms', 'bathrooms', 'lot_size_sqft', 'garage_spaces', 'has_pool', 'has_basement', 'stories']
